# ðŸ”¨ ForgeRL â€” GRPO Training on Google Colab

**Meta PyTorch Ã— HuggingFace OpenEnv Hackathon 2026**

This notebook trains a code LLM with **GRPO** (Group Relative Policy Optimization)
to orchestrate a multi-agent software engineering pipeline using the **ForgeRL** environment.

### What you'll learn
- How ForgeRL wraps a 9-agent SDLC pipeline as an OpenEnv-compatible RL environment
- How to train with TRL's `GRPOTrainer` + Unsloth 4-bit QLoRA on a free T4 GPU
- How to visualize reward improvement and push the trained model to HuggingFace Hub

### Requirements
- **Runtime**: GPU (T4 or better) â€” Runtime â†’ Change runtime type â†’ T4 GPU
- **Time**: ~2 hours for 300 steps (3B model, G=4 on T4)
- **API key**: Optional â€” set `GOOGLE_API_KEY` if you want real LLM sub-agents

---

## 0 Â· Check GPU

In [6]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Go to Runtime â†’ Change runtime type â†’ GPU')
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total:.1f} GB')

Sun Apr 26 00:06:32 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 546.18                 Driver Version: 546.18       CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  | 00000000:01:00.0 Off |                  N/A |
| N/A   45C    P3              13W /  63W |      0MiB /  6141MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 1 Â· Install Dependencies

In [7]:
# Core dependencies
!pip install -q pydantic>=2.0.0 pyyaml python-dotenv rich matplotlib numpy

# HuggingFace ecosystem
!pip install -q transformers>=4.44.0 datasets>=2.20.0 accelerate>=0.33.0 peft>=0.12.0

# TRL â€” GRPO trainer
!pip install -q 'trl>=0.12.0'

# Unsloth â€” 4-bit QLoRA with fast kernels
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# HuggingFace Hub (for pushing the trained model)
!pip install -q huggingface_hub

print('\nâœ“ All dependencies installed')


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'trl": Expected package name at the start of dependency specifier
    'trl
    ^

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



âœ“ All dependencies installed



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 Â· Clone the Repository

In [8]:
import os

REPO_URL = 'https://github.com/rohanjain1648/META-HACKATHON.git'  # â† update this
REPO_DIR = '/content/META-HACKATHON'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Repository contents:')
!ls -la

Already up to date.
Working directory: d:\content\META-HACKATHON
Repository contents:


'ls' is not recognized as an internal or external command,
operable program or batch file.


## 3 Â· Environment Demo (no GPU / no API key required)

In [9]:
import asyncio
from forge_env.environment import ForgeEnvironment
from forge_env.models import ActionType, ForgeAction

# Run in simulated mode â€” no LLM calls, deterministic sub-agent outputs
env = ForgeEnvironment(use_real_llm=False, max_steps=50)

async def run_demo_episode():
    reset_result = await env.reset(tier=1)
    obs = reset_result.observation
    print(f'Episode: {reset_result.info["episode_id"]}')
    print(f'Spec: {reset_result.info["spec_name"]}')
    print(f'Reviewer: {reset_result.info["reviewer"]}')
    print()

    # Heuristic policy: follow the optimal SDLC workflow
    workflow = [
        (ActionType.DELEGATE_INTAKE,    'Requirements analysis'),
        (ActionType.DELEGATE_ARCHITECT, 'System architecture'),
        (ActionType.DELEGATE_PLANNER,   'Task decomposition'),
        (ActionType.APPROVE_PLAN,       'Plan approved'),
        (ActionType.DELEGATE_QA,        'Write TDD tests'),
        (ActionType.DELEGATE_CODER,     'Generate implementation'),
        (ActionType.DELEGATE_OVERSIGHT, 'Quality check'),
        (ActionType.DELEGATE_QA,        'Tests for next task'),
        (ActionType.DELEGATE_CODER,     'Code for next task'),
        (ActionType.DELEGATE_SECURITY,  'Security audit'),
        (ActionType.FINALIZE,           'Finalize project'),
    ]

    rewards = []
    for action_type, reasoning in workflow:
        if obs.current_phase == 'done':
            break
        if action_type.value not in obs.available_actions:
            if obs.available_actions:
                action_type = ActionType(obs.available_actions[0])
        action = ForgeAction(action_type=action_type, reasoning=reasoning)
        result = await env.step(action)
        obs = result.observation
        rewards.append(result.reward)
        print(f'  Step {obs.step_count:2d}: {action_type.value:<26} reward={result.reward:+.3f} | {obs.current_phase}')
        if result.terminated:
            break

    state = env.state
    print(f'\n--- Episode Summary ---')
    print(f'Total reward   : {state.total_reward:.3f}')
    print(f'Task success   : {state.true_test_pass_rate:.1%}')
    print(f'Quality score  : {state.true_code_quality_score:.2f}')
    print(f'Termination    : {state.termination_reason}')
    return rewards

rewards = asyncio.run(run_demo_episode())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ForgeRL Demo â€” Episode Reward', fontweight='bold')

colors = ['#10b981' if r >= 0 else '#ef4444' for r in rewards]
ax1.bar(range(len(rewards)), rewards, color=colors, alpha=0.8)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Step'); ax1.set_ylabel('Reward'); ax1.set_title('Per-Step Rewards')
ax1.grid(True, alpha=0.3)

cumulative = np.cumsum(rewards)
ax2.plot(cumulative, color='#6366f1', linewidth=2)
ax2.fill_between(range(len(cumulative)), cumulative, alpha=0.15, color='#6366f1')
ax2.set_xlabel('Step'); ax2.set_ylabel('Cumulative Reward'); ax2.set_title('Cumulative Reward')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('demo_rewards.png', dpi=150)
plt.show()
print('Saved: demo_rewards.png')

## 4 Â· GRPO Training with Unsloth + TRL

This section trains `Qwen2.5-Coder-3B-Instruct` (or a smaller fallback) using
GRPO with the ForgeRL environment as the reward function.

**Memory guide (T4 = 16 GB):**
| Config | VRAM | Time |
|---|---|---|
| 3B, G=4, 300 steps | ~14 GB | ~2h |
| 1.5B, G=4, 300 steps | ~8 GB | ~1h |
| 3B, G=8, 300 steps | OOM on T4 â€” use A100 | â€” |

In [ ]:
# â”€â”€ Configuration â€” tweak these â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MODEL_NAME      = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'  # smaller = safer on T4
# MODEL_NAME    = 'Qwen/Qwen2.5-Coder-3B-Instruct'    # better, ~14 GB VRAM
OUTPUT_DIR      = './forgerl_outputs'
MAX_STEPS       = 100     # 100 steps â‰ˆ 20 min on T4 for a quick check
# MAX_STEPS     = 300     # full run â‰ˆ 2h
NUM_GENERATIONS = 4       # G: rollouts per prompt (8 = better but needs more VRAM)
BATCH_SIZE      = 1
GRAD_ACCUM      = 4
LEARNING_RATE   = 5e-6
NUM_SPECS       = 20      # training dataset size
PUSH_TO_HUB     = False   # set True after logging in with huggingface-cli login
HF_HUB_REPO     = 'rohanjain1648/forgerl-coder-3b'  # â† update if pushing
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model  : {MODEL_NAME}')
print(f'Steps  : {MAX_STEPS}')
print(f'Output : {OUTPUT_DIR}')

In [ ]:
# â”€â”€ Load model with Unsloth â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from unsloth import FastLanguageModel
import torch

print('Loading model with Unsloth 4-bit QLoRA...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # auto-detect bf16 / fp16
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# â”€â”€ Build training dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, REPO_DIR)

from training.train_forgerl import create_training_dataset, build_prompt_from_observation
from datasets import Dataset

print(f'Building dataset ({NUM_SPECS} specifications)...')
raw = create_training_dataset(num_specs=NUM_SPECS)
hf_dataset = Dataset.from_list([
    {'prompt': item['prompt'], 'spec_text': item['spec_text']}
    for item in raw
])
print(f'Dataset size: {len(hf_dataset)}')
print('Sample prompt (first 400 chars):')
print(hf_dataset[0]['prompt'][:400])

In [ ]:
# â”€â”€ Reward functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from training.train_forgerl import environment_reward_function, format_reward_function

# Quick smoke-test: compute reward for two dummy completions
test_prompts = [hf_dataset[0]['prompt'], hf_dataset[0]['prompt']]
test_completions = [
    'ACTION: delegate_intake\nREASONING: Start here\nPARAMETERS: {}',
    'ACTION: escalate\nREASONING: Cannot proceed\nPARAMETERS: {}',
]
format_rewards = format_reward_function(test_prompts, test_completions)
print('Format reward function âœ“')
print(f'  Good action reward: {format_rewards[0]:.3f}')
print(f'  Escalate reward   : {format_rewards[1]:.3f}')

In [ ]:
# â”€â”€ Configure GRPOTrainer â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from trl import GRPOTrainer, GRPOConfig
import json, time

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_completion_length=512,
    max_prompt_length=1024,
    logging_steps=10,
    save_steps=50,
    report_to='none',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
)

# Reward history for plotting
reward_history = []

class RewardLogger:
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            r = logs.get('reward', logs.get('train/reward', None))
            if r is not None:
                reward_history.append({'step': state.global_step, 'reward': float(r)})
                print(f'  Step {state.global_step:3d}: reward={float(r):.3f}')

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[environment_reward_function, format_reward_function],
    args=training_args,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
)
trainer.add_callback(RewardLogger())
print('GRPOTrainer configured âœ“')

In [ ]:
# â”€â”€ Train â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=' * 60)
print('  Starting GRPO training...')
print(f'  Model          : {MODEL_NAME}')
print(f'  Steps          : {MAX_STEPS}')
print(f'  Generations(G) : {NUM_GENERATIONS}')
print(f'  Reward funcs   : environment + format')
print('=' * 60)

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f'\nTraining complete in {elapsed/60:.1f} min')
print(f'Final step: {train_result.global_step}')
print(f'Final loss: {train_result.training_loss:.4f}')

In [ ]:
# â”€â”€ Save model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
model_save_path = os.path.join(OUTPUT_DIR, 'final_model')
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f'LoRA adapters saved to: {model_save_path}')

# Save reward history
reward_path = os.path.join(OUTPUT_DIR, 'reward_history.json')
with open(reward_path, 'w') as f:
    json.dump(reward_history, f)
print(f'Reward history saved to: {reward_path}')
print(f'\nTotal logged steps: {len(reward_history)}')

## 5 Â· Results â€” Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not reward_history:
    print('No reward history logged â€” run training first')
else:
    steps   = [e['step']   for e in reward_history]
    rewards = [e['reward'] for e in reward_history]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(
        f'ForgeRL GRPO Training â€” {MODEL_NAME.split("/")[-1]}',
        fontsize=14, fontweight='bold'
    )

    # Raw reward
    ax = axes[0]
    ax.plot(steps, rewards, alpha=0.3, color='#6366f1', label='Raw')
    if len(rewards) >= 10:
        w = max(3, len(rewards) // 5)
        smoothed = np.convolve(rewards, np.ones(w)/w, mode='valid')
        ax.plot(steps[w-1:], smoothed, color='#6366f1', linewidth=2, label=f'Smoothed ({w}-step)')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Reward')
    ax.set_title('Mean Episode Reward')
    ax.legend(); ax.grid(True, alpha=0.3)

    # Reward improvement vs baseline
    ax = axes[1]
    baseline = rewards[0] if rewards else 0.14
    deltas = [r - baseline for r in rewards]
    colors = ['#10b981' if d >= 0 else '#ef4444' for d in deltas]
    ax.bar(steps, deltas, color=colors, alpha=0.7)
    ax.axhline(0, color='gray', linestyle='--')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Î” Reward vs Baseline')
    ax.set_title('Improvement over Baseline')
    ax.grid(True, alpha=0.3)

    # Cumulative reward
    ax = axes[2]
    cumulative = np.cumsum(rewards)
    ax.plot(steps, cumulative, color='#10b981', linewidth=2)
    ax.fill_between(steps, cumulative, alpha=0.15, color='#10b981')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Cumulative Reward')
    ax.set_title('Cumulative Training Reward')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    curve_path = os.path.join(OUTPUT_DIR, 'reward_curves.png')
    plt.savefig(curve_path, dpi=150)
    plt.show()
    print(f'Saved: {curve_path}')

    # Summary
    print(f'\nTraining Summary:')
    print(f'  Baseline reward : {rewards[0]:.3f}')
    print(f'  Final reward    : {rewards[-1]:.3f}')
    print(f'  Improvement     : {rewards[-1] - rewards[0]:+.3f}')
    print(f'  Max reward      : {max(rewards):.3f}')

## 6 Â· Evaluate: Before vs. After

Compare baseline model (untrained) vs trained model on 3 episodes.

In [ ]:
# Run the evaluation script if available
import os
eval_script = os.path.join(REPO_DIR, 'training', 'eval_forgerl.py')
if os.path.exists(eval_script):
    !python {eval_script} --model {model_save_path} --episodes 3
else:
    # Manual quick eval using the environment
    import asyncio
    from forge_env.environment import ForgeEnvironment
    from forge_env.models import ActionType, ForgeAction

    async def quick_eval(num_episodes=3):
        env = ForgeEnvironment(use_real_llm=False, max_steps=50)
        all_rewards = []
        for ep in range(num_episodes):
            reset_result = await env.reset(tier=1)
            obs = reset_result.observation
            ep_reward = 0.0
            workflow = [
                ActionType.DELEGATE_INTAKE, ActionType.DELEGATE_ARCHITECT,
                ActionType.DELEGATE_PLANNER, ActionType.APPROVE_PLAN,
                ActionType.DELEGATE_QA, ActionType.DELEGATE_CODER,
                ActionType.DELEGATE_SECURITY, ActionType.FINALIZE,
            ]
            for at in workflow:
                if obs.current_phase == 'done':
                    break
                if at.value not in obs.available_actions:
                    if obs.available_actions:
                        at = ActionType(obs.available_actions[0])
                result = await env.step(ForgeAction(action_type=at, reasoning='eval'))
                obs = result.observation
                ep_reward += result.reward
                if result.terminated:
                    break
            all_rewards.append(ep_reward)
            print(f'Episode {ep+1}: total_reward={ep_reward:.3f}, quality={env.state.true_code_quality_score:.2f}')
        print(f'\nMean reward: {sum(all_rewards)/len(all_rewards):.3f}')
        return all_rewards

    asyncio.run(quick_eval(3))

## 7 Â· (Optional) Push to HuggingFace Hub

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import login
    # Get your token from https://huggingface.co/settings/tokens
    login()  # will prompt for token

    print(f'Pushing model to: {HF_HUB_REPO}')
    model.push_to_hub(HF_HUB_REPO)
    tokenizer.push_to_hub(HF_HUB_REPO)
    print(f'âœ“ Model pushed to: https://huggingface.co/{HF_HUB_REPO}')

    # Optionally push merged 16-bit for inference
    merged_path = os.path.join(OUTPUT_DIR, 'merged_16bit')
    try:
        model.save_pretrained_merged(
            merged_path,
            tokenizer,
            save_method='merged_16bit',
        )
        print(f'âœ“ Merged 16-bit saved to: {merged_path}')
    except Exception as e:
        print(f'Could not merge (adapters still saved): {e}')
else:
    print('PUSH_TO_HUB = False â€” skipping. Set to True and run again to push.')
    print(f'Model saved locally at: {model_save_path}')

---

## Summary

You've successfully:
1. âœ… Explored the ForgeRL environment in simulated mode
2. âœ… Trained a code LLM with GRPO via TRL + Unsloth 4-bit QLoRA
3. âœ… Visualized reward improvement curves
4. âœ… Evaluated before/after performance

### Next steps
- Increase `MAX_STEPS` to 300 for a full training run (~2h on T4)
- Try `NUM_GENERATIONS=8` on an A100 for better GRPO variance reduction
- Set `PUSH_TO_HUB=True` and push your trained model to HF Hub
- Open the [HF Space](https://huggingface.co/spaces/rohanjain1648/forgerl) to demo interactively

### Links
- GitHub: [forgerl](https://github.com/rohanjain1648/META-HACKATHON)
- HF Space: [Interactive Demo](https://huggingface.co/spaces/rohanjain1648/forgerl)
- Docs: `README.md`, `ForgeAI_Detailed_Design_Notes.md`